# Synthetic Eye Symptom Dataset Preparation
Use this notebook to regenerate the synthetic text corpora that feed the multimodal system.

Running the cell below will overwrite the CSV files under `data_preparation/synthetic_data/`.


In [8]:
# Synthetic Symptom Dataset Generator Utilities
import json
import random
from pathlib import Path
import pandas as pd


DISEASE_DATA = {
    "cataract": {
        "name": "Cataract",
        "common_symptoms": [
            "cloudy or blurry vision",
            "faded colors",
            "glare sensitivity",
            "poor night vision",
            "double vision in one eye",
            "frequent prescription changes",
            "yellowing of vision"
        ],
        "templates": [
            "My vision seems {symptom1} and colors look like {symptom2}. Could this be a cataract?",
            "I've been struggling with {symptom1} for {duration}, especially at night.",
            "My father has {symptom1} and now I'm experiencing {symptom2}. Should I be concerned?",
            "Help! I'm seeing halos around lights and have {symptom1}.",
            "Is {symptom1} combined with {symptom2} a sign of cataracts?",
            "I noticed {symptom1} starting {when}. Do I need to see a doctor?",
            "My {relation} eye has {symptom1}. What should I do?",
            "Experiencing {symptom1} and {symptom2}. Need some advice.",
            "How serious is {symptom1} when it's accompanied by {symptom2}?",
            "Can you help? My vision is getting {symptom1} and I also have {symptom2}."
        ],
        "additional_questions": [
            "How long have you noticed the cloudy vision?",
            "Do you have difficulty driving at night?",
            "Are colors appearing faded or yellowed?",
            "Do you see halos around lights?",
            "Has your age exceeded 60 years?"
        ],
        "risk_factors": [
            "age over 60",
            "diabetes",
            "smoking",
            "prolonged sun exposure",
            "eye injury history"
        ],
        "remedies": [
            "Consult an ophthalmologist for comprehensive eye examination",
            "Surgery is the only effective treatment for cataracts",
            "Use brighter lighting for reading",
            "Wear anti-glare sunglasses",
            "Update eyeglass prescription as needed"
        ],
        "urgency": "medium",
        "when_to_see_doctor": "When vision problems interfere with daily activities"
    },
    "diabetic_retinopathy": {
        "name": "Diabetic Retinopathy",
        "common_symptoms": [
            "spots or dark strings floating in vision (floaters)",
            "blurred vision",
            "fluctuating vision",
            "dark or empty areas in vision",
            "vision loss",
            "difficulty seeing colors"
        ],
        "templates": [
            "As a diabetic, I'm seeing {symptom1} and my vision is {symptom2}. Is this diabetic retinopathy?",
            "I have diabetes and have been experiencing {symptom1} for {duration}.",
            "My vision has been {symptom1} and I'm seeing {symptom2}. I'm worried because I'm diabetic.",
            "Help! I have {symptom1} and I'm concerned it's related to my diabetes.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor right away?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice for a diabetic patient.",
            "How serious is {symptom1} with {symptom2} for someone with diabetes?",
            "Can you help? My eye has {symptom1} and I'm also seeing {symptom2}."
        ],
        "additional_questions": [
            "Do you have diabetes? If yes, how long?",
            "Have you noticed sudden vision changes?",
            "Do you see floating spots or cobwebs?",
            "Is your blood sugar well controlled?",
            "When was your last eye examination?"
        ],
        "risk_factors": [
            "diabetes duration",
            "poor blood sugar control",
            "high blood pressure",
            "high cholesterol",
            "pregnancy"
        ],
        "remedies": [
            "Immediate consultation with ophthalmologist",
            "Maintain strict blood sugar control",
            "Control blood pressure and cholesterol",
            "Laser treatment may be required",
            "Regular eye examinations (at least annually)",
            "Anti-VEGF injections may be recommended"
        ],
        "urgency": "high",
        "when_to_see_doctor": "Immediately if you have diabetes and notice vision changes"
    },
    "glaucoma": {
        "name": "Glaucoma",
        "common_symptoms": [
            "gradual loss of peripheral vision",
            "tunnel vision in advanced stages",
            "eye pain",
            "headache",
            "blurred vision",
            "halos around lights",
            "eye redness",
            "nausea and vomiting (acute angle-closure)"
        ],
        "templates": [
            "I've had {symptom1} and {symptom2}. Could it be glaucoma?",
            "I've been experiencing {symptom1} for {duration}. Is this serious?",
            "My friend has {symptom1} and {symptom2}. What could it be?",
            "Help! I have {symptom1} and I'm worried.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and looks {symptom2}."
        ],
        "additional_questions": [
            "Have you noticed loss of peripheral vision?",
            "Do you experience severe eye pain?",
            "Is there a family history of glaucoma?",
            "Have you had eye pressure measurements?",
            "Are you over 40 years old?"
        ],
        "risk_factors": [
            "age over 40",
            "family history",
            "high eye pressure",
            "thin corneas",
            "extreme nearsightedness or farsightedness"
        ],
        "remedies": [
            "See an ophthalmologist immediately",
            "Eye drops to lower eye pressure",
            "Laser treatment (trabeculoplasty)",
            "Surgical intervention if necessary",
            "Regular monitoring of eye pressure",
            "Avoid activities that increase eye pressure"
        ],
        "urgency": "high",
        "when_to_see_doctor": "Immediately, especially if experiencing acute symptoms"
    },
    "normal": {
        "name": "Normal/Healthy Eye",
        "common_symptoms": [
            "clear vision",
            "no pain or discomfort",
            "normal color perception",
            "no floaters or flashes",
            "no redness or irritation",
            "feeling fine",
            "no issues"
        ],
        "templates": [
            "My eyes feel fine, but I wanted to ask if there's anything I should do for my eye health.",
            "I don't have any specific symptoms like {symptom1}, but I work on the computer all day. Any advice?",
            "Just a routine check-up query. My vision is {symptom1} and I feel no {symptom2}. Am I doing okay?",
            "I have {symptom1} and {symptom2}. I think my eyes are healthy, but I want to be sure.",
            "No real problems, just wondering what are the best practices for eye care?",
            "My vision is {symptom1}. Is there anything I can do to keep it that way?",
            "I have a history of eye problems in my family, but I have {symptom1}. What should I do to prevent issues?",
            "I'm just checking in. I have {symptom1} and {symptom2}, which I think is normal.",
            "My eyes feel healthy. I have {symptom1}. Any tips for maintaining good vision?",
            "I have {symptom1} and {symptom2}. Just want to make sure everything is alright."
        ],
        "additional_questions": [
            "Are you experiencing any vision problems?",
            "When was your last eye check-up?",
            "Do you wear corrective lenses?",
            "Any family history of eye diseases?"
        ],
        "risk_factors": [],
        "remedies": [
            "Maintain regular eye examinations",
            "Protect eyes from UV radiation with sunglasses",
            "Take breaks during screen time (20-20-20 rule)",
            "Eat a balanced diet rich in vitamins A, C, and E",
            "Stay hydrated",
            "Avoid smoking"
        ],
        "urgency": "low",
        "when_to_see_doctor": "Annual routine check-up recommended"
    },
    "age_related_macular_degeneration": {
        "name": "Age-related Macular Degeneration (AMD)",
        "common_symptoms": [
            "blurred central vision",
            "straight lines appear wavy",
            "difficulty recognizing faces",
            "dark or empty area in center of vision",
            "decreased brightness of colors",
            "difficulty adapting to low light"
        ],
        "templates": [
            "I'm over 60 and straight lines are starting to look wavy. Could this be AMD?",
            "My central vision is {symptom1} and I'm having {symptom2}. Is this serious?",
            "I'm having trouble recognizing faces and my vision is {symptom1}. What could be wrong?",
            "There's a {symptom1} in the center of my vision. Should I be worried?",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and I'm also seeing {symptom2}."
        ],
        "additional_questions": [
            "Are you over 50 years old?",
            "Do straight lines appear bent or wavy?",
            "Is your central vision becoming blurry?",
            "Do you have difficulty reading?",
            "Is there a family history of macular degeneration?"
        ],
        "risk_factors": [
            "age over 50",
            "smoking",
            "family history",
            "race (more common in Caucasians)",
            "obesity",
            "cardiovascular disease"
        ],
        "remedies": [
            "Consult retina specialist immediately",
            "Anti-VEGF injections for wet AMD",
            "Vitamin supplements (AREDS2 formula)",
            "Quit smoking",
            "Eat leafy green vegetables and fish",
            "Monitor vision with Amsler grid",
            "Low vision aids if needed"
        ],
        "urgency": "high",
        "when_to_see_doctor": "Immediately upon noticing symptoms"
    },
    "hypertensive_retinopathy": {
        "name": "Hypertensive Retinopathy",
        "common_symptoms": [
            "reduced vision",
            "blurred vision",
            "double vision",
            "sudden vision loss",
            "headaches"
        ],
        "templates": [
            "I have high blood pressure and now my vision is {symptom1}. Are these related?",
            "My vision has been {symptom1} and I'm getting {symptom2}. I have hypertension.",
            "I'm experiencing {symptom1} and I'm worried it's because of my high blood pressure.",
            "Help! I have {symptom1} and I'm concerned it's related to my hypertension.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor right away?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice for a hypertensive patient.",
            "How serious is {symptom1} with {symptom2} for someone with high blood pressure?",
            "Can you help? My eye has {symptom1} and I'm also seeing {symptom2}."
        ],
        "additional_questions": [
            "Do you have high blood pressure?",
            "Is your blood pressure controlled?",
            "Have you experienced sudden vision changes?",
            "Do you take blood pressure medications?",
            "When was your last blood pressure check?"
        ],
        "risk_factors": [
            "uncontrolled hypertension",
            "prolonged high blood pressure",
            "cardiovascular disease",
            "kidney disease"
        ],
        "remedies": [
            "Control blood pressure immediately",
            "See both ophthalmologist and primary care physician",
            "Take prescribed antihypertensive medications",
            "Regular blood pressure monitoring",
            "Low-sodium diet",
            "Regular exercise",
            "Stress management"
        ],
        "urgency": "high",
        "when_to_see_doctor": "Immediately if you have hypertension and vision changes"
    },
    "pathological_myopia": {
        "name": "Pathological Myopia",
        "common_symptoms": [
            "severe nearsightedness",
            "progressive vision deterioration",
            "floaters",
            "flashes of light",
            "difficulty seeing distant objects",
            "distorted vision",
            "reduced visual acuity"
        ],
        "templates": [
            "My nearsightedness is getting worse, and I'm seeing {symptom1}. Is this pathological myopia?",
            "I've been experiencing {symptom1} for {duration}. Is this serious?",
            "My friend has {symptom1} and {symptom2}. What could it be?",
            "Help! I have {symptom1} and I'm worried.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and looks {symptom2}."
        ],
        "additional_questions": [
            "Have you been very nearsighted since childhood?",
            "Is your vision getting progressively worse?",
            "Do you see flashes or floaters?",
            "Is there a family history of severe myopia?",
            "Have you experienced retinal complications?"
        ],
        "risk_factors": [
            "high myopia",
            "family history",
            "Asian ethnicity",
            "premature birth",
            "genetic factors"
        ],
        "remedies": [
            "Regular ophthalmologist monitoring",
            "Prescription glasses or contact lenses",
            "Avoid eye strain and excessive near work",
            "Monitor for retinal complications",
            "Consider atropine eye drops (in children)",
            "Orthokeratology lenses",
            "Possible refractive surgery consultation"
        ],
        "urgency": "medium",
        "when_to_see_doctor": "Regular check-ups; immediately if sudden vision changes"
    },
    "conjunctivitis": {
        "name": "Conjunctivitis (Pink Eye)",
        "common_symptoms": [
            "eye redness",
            "itchy eyes",
            "gritty feeling in eyes",
            "discharge from eyes",
            "tearing",
            "crusting of eyelids",
            "light sensitivity"
        ],
        "templates": [
            "My eye is {symptom1} and {symptom2}. Is this pink eye?",
            "I've been experiencing {symptom1} for {duration}. Is this serious?",
            "My friend has {symptom1} and {symptom2}. What could it be?",
            "Help! I have {symptom1} and I'm worried.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and looks {symptom2}."
        ],
        "additional_questions": [
            "Is there discharge from your eye? What color?",
            "Is your eye itchy or burning?",
            "Have you been exposed to someone with pink eye?",
            "Do you have allergies?",
            "Are both eyes affected?"
        ],
        "risk_factors": [
            "exposure to infection",
            "allergies",
            "contact lens wear",
            "touching eyes with unwashed hands"
        ],
        "remedies": [
            "Warm or cold compresses",
            "Artificial tears",
            "Antibiotic eye drops (for bacterial)",
            "Antihistamine drops (for allergic)",
            "Avoid contact lenses during infection",
            "Wash hands frequently",
            "Avoid touching or rubbing eyes"
        ],
        "urgency": "low to medium",
        "when_to_see_doctor": "If symptoms persist beyond 3-5 days or worsen"
    },
    "retinal_detachment": {
        "name": "Retinal Detachment",
        "common_symptoms": [
            "sudden appearance of floaters",
            "flashes of light",
            "shadow or curtain over visual field",
            "sudden blurred vision",
            "gradually reduced peripheral vision"
        ],
        "templates": [
            "I'm seeing {symptom1} and a {symptom2} in my vision. Is this a retinal detachment?",
            "I've been experiencing {symptom1} for {duration}. Is this serious?",
            "My friend has {symptom1} and {symptom2}. What could it be?",
            "Help! I have {symptom1} and I'm worried.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and looks {symptom2}."
        ],
        "additional_questions": [
            "Did you experience sudden onset of floaters?",
            "Do you see flashes of light?",
            "Is there a shadow or curtain in your vision?",
            "Have you had eye surgery recently?",
            "Any history of eye trauma?"
        ],
        "risk_factors": [
            "severe myopia",
            "previous eye surgery",
            "eye injury",
            "family history",
            "previous retinal detachment in other eye"
        ],
        "remedies": [
            "THIS IS AN EMERGENCY - Seek immediate medical attention",
            "Go to emergency room or ophthalmologist immediately",
            "Do not delay treatment",
            "Surgery is required to repair retinal detachment",
            "Avoid activities that cause jarring or pressure"
        ],
        "urgency": "emergency",
        "when_to_see_doctor": "IMMEDIATELY - This is a medical emergency"
    },
    "corneal_ulcer": {
        "name": "Corneal Ulcer",
        "common_symptoms": [
            "severe eye pain",
            "eye redness",
            "excessive tearing",
            "discharge from eye",
            "blurred vision",
            "white spot on cornea",
            "light sensitivity",
            "swollen eyelids"
        ],
        "templates": [
            "I have {symptom1} and {symptom2}. Could this be a corneal ulcer?",
            "I've been experiencing {symptom1} for {duration}. Is this serious?",
            "My friend has {symptom1} and {symptom2}. What could it be?",
            "Help! I have {symptom1} and I'm worried.",
            "Is {symptom1} combined with {symptom2} a sign of {disease_name}?",
            "I noticed {symptom1} {when}. Should I see a doctor?",
            "My {relation} eye has {symptom1}. What to do?",
            "Experiencing {symptom1} and {symptom2}. Need advice.",
            "How serious is {symptom1} with {symptom2}?",
            "Can you help? My eye has {symptom1} and looks {symptom2}."
        ],
        "additional_questions": [
            "Did you have an eye injury recently?",
            "Do you wear contact lenses?",
            "Is there severe eye pain?",
            "Do you see a white spot on your eye?",
            "Is there pus or discharge?"
        ],
        "risk_factors": [
            "contact lens wear",
            "eye injury",
            "dry eyes",
            "immunosuppression",
            "vitamin A deficiency"
        ],
        "remedies": [
            "See ophthalmologist immediately",
            "Antibiotic, antifungal, or antiviral eye drops",
            "Pain medication",
            "Do not wear contact lenses",
            "Do not touch or rub the eye",
            "May require hospitalization for severe cases"
        ],
        "urgency": "high",
        "when_to_see_doctor": "Immediately - can lead to vision loss if untreated"
    }
}


LANG_METADATA = {
    "en": {
        "durations": [
            "a few days",
            "two weeks",
            "a month",
            "yesterday",
            "this morning",
            "last night",
            "for a while",
            "since last week"
        ],
        "whens": [
            "today",
            "yesterday",
            "this morning",
            "last week",
            "recently",
            "suddenly",
            "just now",
            "a moment ago"
        ],
        "relations": [
            "",
            "friend's",
            "mother's",
            "father's",
            "child's",
            "wife's",
            "husband's"
        ]
    },
    "bn": {
        "durations": [
            "কয়েক দিন",
            "দুই সপ্তাহ",
            "এক মাস",
            "গতকাল",
            "আজ সকাল",
            "গত রাত",
            "কিছুক্ষণ ধরে",
            "গত সপ্তাহ থেকে"
        ],
        "whens": [
            "আজ",
            "গতকাল",
            "আজ সকাল",
            "গত সপ্তাহ",
            "সাম্প্রতিক",
            "হঠাৎ",
            "এইমাত্র",
            "একটু আগে"
        ],
        "relations": [
            "",
            "বন্ধুর",
            "মায়ের",
            "বাবার",
            "সন্তানের",
            "স্ত্রীর",
            "স্বামীর"
        ]
    }
}


DISEASE_TRANSLATIONS = {
    "cataract": {
        "name": "ছানি (Cataract)",
        "common_symptoms": [
            "দৃষ্টিতে ঝাপসা বা কুয়াশা",
            "রঙ ফ্যাকাসে দেখা",
            "আলোর প্রতি অতিরিক্ত সংবেদনশীলতা",
            "রাতে দেখতে সমস্যা",
            "এক চোখে দ্বৈত দেখা",
            "বারবার চশমার পাওয়ার বদলানো",
            "দৃষ্টিতে হলুদাভ ভাব"
        ],
        "templates": [
            "আমার দৃষ্টি {symptom1} এবং রঙগুলো {symptom2} লাগছে। এটা কি ছানি হতে পারে?",
            "আমি {duration} ধরে {symptom1} অনুভব করছি, বিশেষ করে রাতে।",
            "আমার বাবার {symptom1} ছিল এবং এখন আমার {symptom2} হচ্ছে। আমার কি চিন্তা করা উচিত?",
            "সাহায্য করুন! আমি আলোর চারপাশে বলয় দেখছি এবং আমার {symptom1} আছে।",
            "{symptom1} এবং {symptom2} একসাথে থাকা কি ছানির লক্ষণ?",
            "আমি {when} থেকে {symptom1} লক্ষ্য করছি। আমার কি ডাক্তার দেখানো উচিত?",
            "আমার {relation} চোখে {symptom1} হয়েছে। কী করব?",
            "{symptom1} এবং {symptom2} অনুভব করছি। কিছু পরামর্শ প্রয়োজন।",
            "{symptom1} এর সাথে {symptom2} থাকা কতটা গুরুতর?",
            "আপনি কি সাহায্য করতে পারেন? আমার দৃষ্টি {symptom1} হয়ে যাচ্ছে এবং সাথে {symptom2} আছে।"
        ]
    },
    "diabetic_retinopathy": {
        "name": "ডায়াবেটিক রেটিনোপ্যাথি",
        "common_symptoms": [
            "দৃষ্টিতে ঝাপসা",
            "চোখের সামনে কালো দাগ ভাসা",
            "রাতে দেখতে অসুবিধা",
            "রঙ পার্থক্য করতে সমস্যা",
            "হঠাৎ দৃষ্টি ঝাপসা হওয়া",
            "চোখে অন্ধকার অংশ দেখা",
            "দৃষ্টিক্ষমতা কমে যাওয়া",
            "পড়তে কষ্ট হওয়া"
        ],
        "templates": [
            "আমি ডায়াবেটিক এবং {symptom1} সঙ্গে {symptom2} হচ্ছে। কী করা উচিত?",
            "ডায়াবেটিসের কারণে {symptom1} ও {symptom2} অনুভব করছি।",
            "{duration} ধরে {symptom1} আর {symptom2} হচ্ছে, আমি ডায়াবেটিক।",
            "{when} থেকে {symptom1} ও {symptom2} হচ্ছে। এটা কি {disease_name}?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} আছে, আমরা ডায়াবেটিস রোগী।"
        ]
    },
    "glaucoma": {
        "name": "গ্লুকোমা",
        "common_symptoms": [
            "পাশের দৃষ্টি ধীরে ধীরে কমে যাওয়া",
            "টানেল ভিশন",
            "চোখে ব্যথা",
            "মাথাব্যথা",
            "আলোর চারপাশে হলা দেখা",
            "চোখ লাল হওয়া",
            "বমি ভাব",
            "দৃষ্টিতে ঝাপসা"
        ],
        "templates": [
            "{symptom1} এবং {symptom2} অনুভব করছি। এটা কি {disease_name}?",
            "আমার কাছে {symptom1} ও {symptom2} আছে, কী করা দরকার?",
            "{when} থেকে {symptom1} ও {symptom2} হচ্ছে।",
            "আমার {relation} চোখে {symptom1} ও {symptom2} হয়েছে, পরামর্শ দিন।",
            "{duration} ধরে {symptom1} হচ্ছে এবং এখন {symptom2} শুরু হয়েছে।"
        ]
    },
    "normal": {
        "name": "স্বাভাবিক চোখ",
        "common_symptoms": [
            "কোনো অভিযোগ নেই",
            "দৃষ্টি স্বাভাবিক",
            "চোখে কোনো ব্যথা নেই",
            "রঙ ঠিকঠাক দেখা যায়",
            "ফ্লোটার বা ঝলক নেই",
            "ভালো লাগছে",
            "কোনো সমস্যা নেই"
        ],
        "templates": [
            "আমার চোখ ঠিক আছে, কিন্তু চোখের স্বাস্থ্যের জন্য আমার কী করা উচিত?",
            "আমার {symptom1} এর মতো কোনো নির্দিষ্ট লক্ষণ নেই, কিন্তু আমি সারাদিন কম্পিউটারে কাজ করি। কোনো পরামর্শ আছে?",
            "শুধু একটি রুটিন প্রশ্ন। আমার দৃষ্টি {symptom1} এবং আমি কোনো {symptom2} অনুভব করছি না। আমি কি ঠিক আছি?",
            "আমার {symptom1} এবং {symptom2} আছে। আমার মনে হয় চোখ সুস্থ, কিন্তু আমি নিশ্চিত হতে চাই।",
            "কোনো আসল সমস্যা নেই, শুধু চোখের যত্নের সেরা উপায়গুলো কী তা জানতে চাই?",
            "আমার দৃষ্টি {symptom1}। এটিকে এরকম রাখতে আমি কী করতে পারি?",
            "আমার পরিবারে চোখের সমস্যার ইতিহাস আছে, কিন্তু আমার {symptom1}। সমস্যা প্রতিরোধে আমার কী করা উচিত?",
            "আমি শুধু খোঁজ নিচ্ছি। আমার {symptom1} এবং {symptom2} আছে, যা আমার মনে হয় স্বাভাবিক।",
            "আমার চোখ সুস্থ মনে হচ্ছে। আমার {symptom1} আছে। ভালো দৃষ্টি বজায় রাখার জন্য কোনো টিপস আছে?",
            "আমার {symptom1} এবং {symptom2} আছে। শুধু নিশ্চিত হতে চাই যে সবকিছু ঠিক আছে।"
        ]
    },
    "age_related_macular_degeneration": {
        "name": "বয়সজনিত ম্যাকুলার ক্ষয়",
        "common_symptoms": [
            "মাঝের দৃষ্টি ঝাপসা",
            "সোজা লাইন বেঁকে দেখা",
            "মুখ চিনতে সমস্যা",
            "কেন্দ্রে অন্ধকার দাগ",
            "রঙ ফিকে দেখায়",
            "অন্ধকারে দেখতে কষ্ট"
        ],
        "templates": [
            "আমি {symptom1} ও {symptom2} অনুভব করছি। এটা কি {disease_name}?",
            "{duration} ধরে আমার দৃষ্টিতে {symptom1} এবং {symptom2}।",
            "{when} থেকে {symptom1} ও {symptom2} বাড়ছে। কী করব?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} আছে। পরামর্শ দিন।",
            "দৃষ্টিতে {symptom1} অনুভব করছি এবং সঙ্গে {symptom2} আছে।"
        ]
    },
    "hypertensive_retinopathy": {
        "name": "উচ্চ রক্তচাপজনিত রেটিনোপ্যাথি",
        "common_symptoms": [
            "দৃষ্টিশক্তি কমে যাওয়া",
            "দৃষ্টি ঝাপসা",
            "দ্বৈত দৃষ্টি",
            "হঠাৎ দৃষ্টি হারানো",
            "মাথাব্যথা"
        ],
        "templates": [
            "উচ্চ রক্তচাপের সঙ্গে {symptom1} ও {symptom2} হচ্ছে। এখন কী করা উচিত?",
            "{duration} ধরে {symptom1} অনুভব করছি এবং আজ {symptom2} হয়েছে।",
            "{when} থেকে {symptom1} ও {symptom2} হচ্ছে। এটা কি {disease_name}?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} আছে এবং আমরা হাইপারটেনসিভ।",
            "আমি {symptom1} অনুভব করছি, পাশাপাশি {symptom2}। সাহায্য করুন।"
        ]
    },
    "pathological_myopia": {
        "name": "প্যাথলজিক্যাল মায়োপিয়া",
        "common_symptoms": [
            "দূরের বস্তু ঝাপসা দেখা",
            "দৃষ্টি দ্রুত খারাপ হওয়া",
            "চোখে ফ্লোটার দেখা",
            "আলোর ঝলক দেখা",
            "দূর থেকে পড়তে সমস্যা",
            "চোখ খুব ক্লান্ত লাগে"
        ],
        "templates": [
            "{duration} ধরে {symptom1} ও {symptom2} হচ্ছে। এটা কি {disease_name}?",
            "আমার {relation} চোখে {symptom1} এবং {symptom2} দেখা যাচ্ছে।",
            "{when} থেকে {symptom1} অনুভব করছি এবং এখন {symptom2}।",
            "{symptom1} ও {symptom2} থাকার ফলে দৈনন্দিন কাজ কঠিন হচ্ছে।",
            "আমি {symptom1} নিয়ে ভুগছি এবং পাশাপাশি {symptom2} আছে।"
        ]
    },
    "conjunctivitis": {
        "name": "কনজাঙ্কটিভাইটিস (পিঙ্ক আই)",
        "common_symptoms": [
            "চোখ লাল হওয়া",
            "চোখ চুলকানো",
            "চোখে বালি বালি লাগা",
            "চোখ থেকে পানি পড়া",
            "চোখে পুঁজ বা স্রাব হওয়া",
            "আলোতে অস্বস্তি",
            "চোখের পাতায় খোসা জমা"
        ],
        "templates": [
            "{symptom1} ও {symptom2} হচ্ছে। এটা কি {disease_name}?",
            "{duration} ধরে {symptom1} অনুভব করছি এবং {symptom2} দেখা যাচ্ছে।",
            "{when} থেকে {symptom1} ও {symptom2} আছে। কী করা উচিত?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} হয়েছে। চিকিৎসা দরকার কি?",
            "আমি {symptom1} অনুভব করছি এবং সঙ্গে {symptom2} আছে। সাহায্য করুন।"
        ]
    },
    "retinal_detachment": {
        "name": "রেটিনার বিচ্ছেদ",
        "common_symptoms": [
            "হঠাৎ ফ্লোটার দেখা",
            "আলোর ঝলক দেখা",
            "চোখের সামনে পর্দা নামা",
            "হঠাৎ দৃষ্টি ঝাপসা হওয়া",
            "পাশের দৃষ্টি কমে যাওয়া"
        ],
        "templates": [
            "{symptom1} এবং {symptom2} হচ্ছে। এটা কি জরুরি অবস্থা?",
            "{duration} ধরে {symptom1} অনুভব করছি এবং এখন {symptom2} আছে।",
            "{when} থেকে {symptom1} ও {symptom2} হচ্ছে। আমি কী করব?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} দেখা যাচ্ছে। এটা কি {disease_name}?",
            "হঠাৎ {symptom1} শুরু হয়েছে এবং সঙ্গে {symptom2} আছে। সাহায্য করুন।"
        ]
    },
    "corneal_ulcer": {
        "name": "করনিয়াল আলসার",
        "common_symptoms": [
            "চোখে তীব্র ব্যথা",
            "চোখ লাল হওয়া",
            "চোখ থেকে পানি পড়া",
            "দৃষ্টি ঝাপসা",
            "করনিয়ায় সাদা দাগ",
            "আলোতে ব্যথা",
            "চোখের পাতায় ফোলা"
        ],
        "templates": [
            "{symptom1} ও {symptom2} অনুভব করছি। এখন কী করা উচিত?",
            "{duration} ধরে {symptom1} আছে এবং আজ {symptom2} শুরু হয়েছে।",
            "{when} থেকে {symptom1} হচ্ছে, সঙ্গে {symptom2}। এটা কি {disease_name}?",
            "আমার {relation} চোখে {symptom1} ও {symptom2} হয়েছে। দয়া করে পরামর্শ দিন।",
            "আমি {symptom1} অনুভব করছি এবং এখন {symptom2} রয়েছে। সাহায্য চাই।"
        ]
    }
}


def generate_patient_queries(disease_key, disease_info, num_samples=50, languages=None):
    """Generate realistic patient queries for each disease in multiple languages."""
    languages = languages or ["en", "bn"]
    queries = []
    for lang in languages:
        symptoms = []
        templates = []
        language_name = ""

        if lang == "en":
            language_name = disease_info.get("name", "")
            symptoms = disease_info.get("common_symptoms", [])
            templates = disease_info.get("templates", [])
        else:
            translation = DISEASE_TRANSLATIONS.get(disease_key, {})
            language_name = translation.get("name", "")
            symptoms = translation.get("common_symptoms", [])
            templates = translation.get("templates", [])

        if len(symptoms) < 1 or not templates:
            continue

        config = LANG_METADATA.get(lang, LANG_METADATA["en"])
        for _ in range(num_samples):
            template = random.choice(templates)
            
            # Vary the number of symptoms from 1 to 2
            num_symptoms_to_pick = random.randint(1, min(2, len(symptoms)))
            picked_symptoms = random.sample(symptoms, num_symptoms_to_pick)
            
            symptom1 = picked_symptoms[0]
            symptom2 = picked_symptoms[1] if len(picked_symptoms) > 1 else random.choice(symptoms)

            query_text = template.format(
                symptom1=symptom1,
                symptom2=symptom2,
                duration=random.choice(config["durations"]),
                when=random.choice(config["whens"]),
                disease_name=language_name,
                relation=random.choice(config["relations"])
            )
            queries.append({
                "disease": disease_key,
                "disease_name": language_name,
                "query": query_text,
                "symptoms_mentioned": picked_symptoms,
                "urgency": disease_info["urgency"],
                "language": lang
            })
    return queries


def generate_qa_pairs(disease_key, disease_info, num_samples=30):
    """Generate question-answer pairs for conversational training."""
    qa_pairs = []
    for question in disease_info["additional_questions"]:
        qa_pairs.append({
            "disease": disease_key,
            "question": question,
            "answer": random.choice([
                "Yes", "Yes, for several years", "Yes, it's been ongoing",
                "Yes, I noticed that", "Absolutely", "Yes, that's correct"
            ]),
            "confirms_diagnosis": True
        })
        qa_pairs.append({
            "disease": disease_key,
            "question": question,
            "answer": random.choice([
                "No", "No, I don't think so", "Not really",
                "I don't have that", "No, that's not the case"
            ]),
            "confirms_diagnosis": False
        })
    return qa_pairs


def generate_complete_consultations(num_consultations=20):
    """Generate multi-turn consultation scenarios."""
    consultations = []
    for disease_key, disease_info in DISEASE_DATA.items():
        for _ in range(num_consultations):
            symptoms = random.sample(
                disease_info["common_symptoms"],
                min(3, len(disease_info["common_symptoms"]))
            )
            consultation = {
                "disease": disease_key,
                "disease_name": disease_info["name"],
                "initial_complaint": f"I have {', '.join(symptoms[:2])}",
                "symptoms": symptoms,
                "conversation": [],
                "diagnosis": disease_info["name"],
                "remedies": disease_info["remedies"],
                "urgency": disease_info["urgency"]
            }
            for i, question in enumerate(random.sample(
                disease_info["additional_questions"],
                min(3, len(disease_info["additional_questions"]))
            )):
                consultation["conversation"].append({
                    "turn": i + 1,
                    "doctor_question": question,
                    "patient_answer": random.choice([
                        "Yes", "Yes, that's right", "No", "Not really"
                    ])
                })
            consultations.append(consultation)
    return consultations


def generate_synthetic_symptom_datasets(output_dir=Path("data_preparation/synthetic_data"), languages=None):
    """Persist synthetic datasets (English + optional Bangla) to disk."""
    if languages is None:
        languages = ["en", "bn"]
    output_dir.mkdir(parents=True, exist_ok=True)
    print("Generating disease information database...")
    with open(output_dir / "disease_database.json", "w") as f:
        json.dump(DISEASE_DATA, f, indent=2)
    print("Generating patient queries...")
    all_queries = []
    for disease_key, disease_info in DISEASE_DATA.items():
        all_queries.extend(generate_patient_queries(disease_key, disease_info, num_samples=500, languages=languages))
    df_queries = pd.DataFrame(all_queries)
    df_queries.to_csv(output_dir / "patient_queries.csv", index=False)
    print("Generating consultation scenarios (in-memory)...")
    consultations = generate_complete_consultations()
    print("\n✓ Generated datasets:")
    print(f"  - Disease database: {len(DISEASE_DATA)} diseases")
    print(f"  - Patient queries: {len(all_queries)} samples")
    print(f"  - Consultations: {len(consultations)} scenarios (in-memory)")
    print(f"\nFiles saved to: {output_dir}")
    return {
        "disease_database": output_dir / "disease_database.json",
        "patient_queries_csv": output_dir / "patient_queries.csv",
        "consultations": consultations
    }


def generate_symptom_dataset_cli():
    """Notebook-friendly entry point mimicking script execution."""
    return generate_synthetic_symptom_datasets()


if __name__ == "__main__":
    generate_synthetic_symptom_datasets()


Generating disease information database...
Generating patient queries...
Generating consultation scenarios (in-memory)...

✓ Generated datasets:
  - Disease database: 10 diseases
  - Patient queries: 10000 samples
  - Consultations: 200 scenarios (in-memory)

Files saved to: data_preparation\synthetic_data


In [9]:
import os

# Your dataset path
root_dir = r"C:\Users\ajoys\OneDrive\Desktop\Eyediscease\data\eye_images"

# Characters to remove or replace
forbidden_chars = ["[", "]"]

for root, dirs, files in os.walk(root_dir, topdown=False):
    # Rename files
    for name in files:
        new_name = name
        for ch in forbidden_chars:
            new_name = new_name.replace(ch, "_")
        if new_name != name:
            old_path = os.path.join(root, name)
            new_path = os.path.join(root, new_name)
            os.rename(old_path, new_path)
            print(f"Renamed file: {old_path} -> {new_path}")

    # Rename directories
    for name in dirs:
        new_name = name
        for ch in forbidden_chars:
            new_name = new_name.replace(ch, "_")
        if new_name != name:
            old_path = os.path.join(root, name)
            new_path = os.path.join(root, new_name)
            os.rename(old_path, new_path)
            print(f"Renamed folder: {old_path} -> {new_path}")

print("✅ All forbidden characters have been removed.")


Renamed folder: C:\Users\ajoys\OneDrive\Desktop\Eyediscease\data\eye_images\Central Serous Chorioretinopathy [Color Fundus] -> C:\Users\ajoys\OneDrive\Desktop\Eyediscease\data\eye_images\Central Serous Chorioretinopathy _Color Fundus_
✅ All forbidden characters have been removed.
